<a href="https://colab.research.google.com/github/feliciadominic14-create/Homework/blob/main/Linear%26Logical_Regression_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np

#Linear Regression

In [17]:
df_lin = pd.DataFrame({
                        'hours_studied': [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 6.0, 6.5, 7.5],
                        'exam_score':    [42.0, 48.0, 51.0, 56.0, 61.0, 65.0, 70.0, 74.0, 78.0, 85.0, 89.0, 95.0]
                      })

In [18]:
class LinearRegression:
    def __init__(self, lr=0.01, epochs=10000):
        self.lr = lr
        self.epochs = epochs
        self.w = 0.0
        self.b = 0.0

    def fit(self, X, y):
        X = np.asarray(X, dtype=float).flatten()
        y = np.asarray(y, dtype=float).flatten()
        n = len(X)
        for _ in range(self.epochs):
            y_pred = self.w * X + self.b
            dw = (-2.0 / n) * np.sum(X * (y - y_pred))
            db = (-2.0 / n) * np.sum(y - y_pred)
            self.w -= self.lr * dw
            self.b -= self.lr * db
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float).flatten()
        return self.w * X + self.b

In [19]:
lin_model = LinearRegression(lr=0.01, epochs=10000)
lin_model.fit(df_lin['hours_studied'], df_lin['exam_score'])
lin_preds = lin_model.predict(df_lin['hours_studied'])

In [20]:
print("w:", round(lin_model.w, 4), "| b:", round(lin_model.b, 4))
print("Predictions:", np.round(lin_preds, 2))

w: 8.237 | b: 35.5718
Predictions: [43.81 47.93 52.05 56.16 60.28 64.4  68.52 72.64 76.76 84.99 89.11 97.35]


#Logistic Regression

In [21]:
df_log = pd.DataFrame({
    'hours_studied': [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.5, 7.5],
    'passed':        [0,   0,   0,   0,   0,   1,   0,   1,   1,   1,   1,   1]
})

In [22]:
class LogisticRegression:
    def __init__(self, lr=0.2, epochs=10000):
        self.lr = lr
        self.epochs = epochs
        self.w = 0.0
        self.b = 0.0

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y):
        X = np.asarray(X, dtype=float).flatten()
        y = np.asarray(y, dtype=float).flatten()
        n = len(X)
        for _ in range(self.epochs):
            z = self.w * X + self.b
            y_hat = self._sigmoid(z)
            dw = (1.0 / n) * np.sum((y_hat - y) * X)
            db = (1.0 / n) * np.sum(y_hat - y)
            self.w -= self.lr * dw
            self.b -= self.lr * db
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float).flatten()
        return self._sigmoid(self.w * X + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

In [23]:
log_model = LogisticRegression(lr=0.2, epochs=10000)
log_model.fit(df_log['hours_studied'], df_log['passed'])
log_probs = log_model.predict_proba(df_log['hours_studied'])
log_preds = log_model.predict(df_log['hours_studied'])

In [24]:
acc = np.mean(log_preds == df_log['passed'])
eps = 1e-15
p_clip = np.clip(log_probs, eps, 1 - eps)
bce = -np.mean(df_log['passed'] * np.log(p_clip) + (1 - df_log['passed']) * np.log(1 - p_clip))

In [25]:
print(f"Learned parameters -> w: {log_model.w:.4f}, b: {log_model.b:.4f}")
print(f"Accuracy: {acc*100:.2f}% | Binary Cross-Entropy Loss: {bce:.4f}")
results_log = pd.DataFrame({
    'hours_studied': df_log['hours_studied'],
    'actual_pass': df_log['passed'],
    'prob_pass': np.round(log_probs, 3),
    'pred_pass': log_preds
})
print(results_log.to_string(index=False))

Learned parameters -> w: 2.5803, b: -9.6781
Accuracy: 83.33% | Binary Cross-Entropy Loss: 0.2090
 hours_studied  actual_pass  prob_pass  pred_pass
           1.0            0      0.001          0
           1.5            0      0.003          0
           2.0            0      0.011          0
           2.5            0      0.038          0
           3.0            0      0.126          0
           3.5            1      0.344          0
           4.0            0      0.655          1
           4.5            1      0.874          1
           5.0            1      0.962          1
           5.5            1      0.989          1
           6.5            1      0.999          1
           7.5            1      1.000          1
